The notebook to create the harness, prompt set, and analysis pipeline.

In [1]:
%%bash
mkdir -p promptset
curl -sL -o promptset/xstest.csv   https://raw.githubusercontent.com/paul-rottger/xstest/main/xstest_prompts.csv
curl -sL -o promptset/advbench.csv https://raw.githubusercontent.com/llm-attacks/llm-attacks/main/data/advbench/harmful_behaviors.csv
curl -sL -o promptset/alpaca.json  https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json

In [2]:
!python build_promptset.py

seed=20260611  total=200
  benign             60
  borderline_safe    50
  borderline_unsafe  40
  harmful            50
wrote prompts_labeled.tsv and prompts_provenance.tsv

NOTE for paper: prompts sampled from XSTest (CC-BY-4.0; Roettger et al. 2024), AdvBench (Zou et al. 2023), and Alpaca (Taori et al. 2023). Unsafe prompts are used only as classifier inputs; no harmful content is generated.


In [3]:
from huggingface_hub import login

login(token="----------------")  # your HF token

In [4]:
!pip install vllm

INFO: pip is looking at multiple versions of model-hosting-container-standards to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of cuda-tile[tileiras] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 748.7/748.7 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
!pip install --upgrade transformers

INFO: pip is looking at multiple versions of tokenizers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 162.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 156.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 109.4 MB/s eta 0:00:00
  Attempting uninstall: regex
    Found existing installation: regex 2025.9.1
    Uninstalling regex-2025.9.1:
      Successfully uninstalled regex-2025.9.1
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.1.9
    Uninstalling hf-xet-1.1.9:
      Successfully uninstalled hf-xet-1.1.9
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2

In [8]:
vllm.terminate()   # stop the guard8b/guard1b server

In [6]:
import subprocess, os
env = {**os.environ,
       "VLLM_USE_FLASHINFER_SAMPLER": "0",
       "VLLM_ATTENTION_BACKEND": "FLASH_ATTN"}
vllm = subprocess.Popen(
    ["python","-m","vllm.entrypoints.openai.api_server",
     "--model","meta-llama/Llama-Guard-3-1B","--served-model-name","guard1b",
     "--port","8000","--max-model-len","2048",
     "--gpu-memory-utilization","0.90","--enforce-eager"],
    stdout=open("vllm.log","w"), stderr=subprocess.STDOUT, env=env)
print("vLLM launching, pid", vllm.pid)

vLLM launching, pid 674


In [6]:
import subprocess, os
env = {**os.environ,
       "VLLM_USE_FLASHINFER_SAMPLER": "0",
       "VLLM_ATTENTION_BACKEND": "FLASH_ATTN"}
vllm = subprocess.Popen(
    ["python","-m","vllm.entrypoints.openai.api_server",
     "--model","meta-llama/Llama-Guard-3-8B","--served-model-name","guard8b",
     "--port","8000","--max-model-len","2048",
     "--gpu-memory-utilization","0.90","--enforce-eager"],
    stdout=open("vllm.log","w"), stderr=subprocess.STDOUT, env=env)

print("vLLM launching, pid", vllm.pid)

vLLM launching, pid 283


In [7]:
import time, urllib.request
for _ in range(90):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2)
        print("ready"); break
    except Exception:
        time.sleep(2)

ready


In [8]:
%%bash

curl http://localhost:8000/v1/models

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   468  100   468    0     0   115k      0 --:--:-- --:--:-- --:--:--  152k


{"object":"list","data":[{"id":"guard8b","object":"model","created":1781430605,"owned_by":"vllm","root":"meta-llama/Llama-Guard-3-8B","parent":null,"max_model_len":2048,"permission":[{"id":"modelperm-91228952804ad84a","object":"model_permission","created":1781430605,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [10]:
!python sweep.py --gpu A100_40GB --model guard8b --base-url http://localhost:8000 --prompts prompts_labeled.tsv --max-tokens 16 --rung-seconds 20 --outdir results

[preflight] gpu=A100_40GB vram=40GB price=$2.1/hr nvml=on

=== A100_40GB x guard8b  (max_tokens=16) ===
  c=   1  req/s=5.75  tok/s=24.4  ttft_p95=77ms  e2el_p95=236ms  J/tok=5.2401  good=5.75
  c=   2  req/s=10.22  tok/s=43.0  ttft_p95=100ms  e2el_p95=262ms  J/tok=3.3328  good=10.22
  c=   4  req/s=20.787  tok/s=87.1  ttft_p95=98ms  e2el_p95=254ms  J/tok=1.5364  good=20.787
  c=   8  req/s=41.413  tok/s=174.3  ttft_p95=98ms  e2el_p95=254ms  J/tok=0.894  good=41.413
  c=  16  req/s=81.564  tok/s=342.9  ttft_p95=99ms  e2el_p95=256ms  J/tok=0.5113  good=81.564
  c=  32  req/s=140.802  tok/s=591.4  ttft_p95=150ms  e2el_p95=321ms  J/tok=0.3142  good=140.802
  c=  64  req/s=228.703  tok/s=959.8  ttft_p95=204ms  e2el_p95=379ms  J/tok=0.2143  good=213.107
  c= 128  req/s=283.405  tok/s=1189.2  ttft_p95=391ms  e2el_p95=566ms  J/tok=0.1845  good=6.827
  c= 256  req/s=329.111  tok/s=1381.3  ttft_p95=691ms  e2el_p95=953ms  J/tok=0.1681  good=0.0
  saved -> results/A100_40GB_guard8b_mt16.json
  qp

In [11]:
!python sweep.py --gpu A100_40GB --model guard8b --base-url http://localhost:8000 --prompts prompts_labeled.tsv --max-tokens 600 --rung-seconds 20 --outdir results --force-generate

[preflight] gpu=A100_40GB vram=40GB price=$2.1/hr nvml=on

=== A100_40GB x guard8b  (max_tokens=600)  [FORCE-GEN: ignore_eos, min_tokens=max] ===
  c=   1  req/s=0.053  tok/s=32.0  ttft_p95=84ms  e2el_p95=18952ms  J/tok=4.0671  good=0.0
  c=   2  req/s=0.106  tok/s=63.4  ttft_p95=110ms  e2el_p95=18970ms  J/tok=2.0006  good=0.0
  c=   4  req/s=0.213  tok/s=127.7  ttft_p95=115ms  e2el_p95=18942ms  J/tok=1.0079  good=0.0
  c=   8  req/s=0.423  tok/s=253.7  ttft_p95=124ms  e2el_p95=18937ms  J/tok=0.5341  good=0.0
  c=  16  req/s=0.78  tok/s=468.0  ttft_p95=152ms  e2el_p95=20514ms  J/tok=0.2859  good=0.0
  c=  32  req/s=1.595  tok/s=957.2  ttft_p95=195ms  e2el_p95=20058ms  J/tok=0.1842  good=0.0
  c=  64  req/s=3.131  tok/s=1878.9  ttft_p95=286ms  e2el_p95=20435ms  J/tok=0.1118  good=0.0
  c= 128  req/s=5.776  tok/s=3465.8  ttft_p95=408ms  e2el_p95=22154ms  J/tok=0.0733  good=0.0
  c= 256  req/s=10.431  tok/s=6258.8  ttft_p95=794ms  e2el_p95=24534ms  J/tok=0.0567  good=0.0
  saved -> result

In [12]:
!python eval_quality.py --model guard8b --base-url http://localhost:8000 --prompts prompts_labeled.tsv --max-tokens 64 --concurrency 8 --outdir results

{
  "model": "guard8b",
  "max_tokens": 64,
  "n": 200,
  "n_error": 0,
  "precision": 0.9873417721518988,
  "recall": 0.8666666666666667,
  "f1": 0.9230769230769232,
  "fpr_benign": 0.0,
  "fpr_borderline_safe": 0.02,
  "per_category_acc": {
    "harmful": 0.96,
    "borderline_unsafe": 0.75,
    "benign": 1.0,
    "borderline_safe": 0.98
  }
}


The experimentation for guard1b

In [ ]:
!python sweep.py --gpu A100_40GB --model guard1b --base-url http://localhost:8000 --prompts prompts_labeled.tsv --max-tokens 16 --rung-seconds 20 --outdir results

In [ ]:
!python sweep.py --gpu A100_40GB --model guard1b --base-url http://localhost:8000 --prompts prompts_labeled.tsv --max-tokens 600 --rung-seconds 20 --outdir results --force-generate

In [ ]:
!python eval_quality.py --model guard1b --base-url http://localhost:8000 --prompts prompts_labeled.tsv --max-tokens 64 --concurrency 8 --outdir results

In [ ]:
!python api_probe.py --endpoints endpoints.json --prompts prompts_labeled.tsv --concurrency 4 --repeats 5 --outdir results

Run these scripts after running all the GPUs

In [13]:
!python aggregate.py

GPU          model     max_tok   $/hr  vram  qps/rep   lat_s    tok/s    J/tok  goodput
------------------------------------------------------------------------------------------------
A100_40GB    guard1b       600   2.10    40   19.804   7.46911882.100    0.016    0.000
A100_40GB    guard1b        16   2.10    40  436.642   0.060 1879.900    0.056   57.857
A100_40GB    guard8b        16   2.10    40  329.111   0.137 1381.300    0.168    0.000
A100_40GB    guard8b       600   2.10    40   10.431  18.747 6258.800    0.057    0.000
A100_80GB    guard1b       600   2.50    80   17.124   7.80710274.300    0.018    0.000
A100_80GB    guard1b        16   2.50    80  350.758   0.065 1509.900    0.069    1.381
A100_80GB    guard8b        16   2.50    80  302.764   0.142 1270.800    0.177    0.000
A100_80GB    guard8b       600   2.50    80   10.583  18.677 6349.900    0.057    0.000
L40S         guard1b       600   1.95    48   31.571   3.61818942.300    0.017    0.000
L40S         guard1b   

In [14]:
!python make_figures.py

loaded 16 cells, 0 N/A, 0 APIs, 2 quality files
  wrote figures/fig1_cost_vs_scale.pdf/.png
  wrote figures/fig2_saturation.pdf/.png
  wrote figures/fig3_energy.pdf/.png
  wrote figures/fig4_pareto.pdf/.png
  wrote figures/table_feasibility.csv


In [15]:
!python pilot_cost_model.py

MECHANISM                                 lat   SLO   API$/1M  SH floor  cross qps
------------------------------------------------------------------------------
Provider API (moderation)               0.10s  PASS     30.0       --        --
Small guard 1B (self-host or API)       0.04s  PASS      8.8      3.5        32
LLM-as-judge 8-70B                      5.00s  FAIL    252.4    100.7         3
Inference-time scaling (deliberative)   9.00s  FAIL    568.4    268.5         1

$/1M spread across classes (best floor -> worst API): 164x
Interactive-SLO (200ms) survivors: ['Provider API (moderation)', 'Small guard 1B (self-host or API)']

Saved chart -> pilot_cost_curves.png
